Some of the images in the lofar dataset are empty but we have the labels for those images as well so while processing the data, make sure remove the images which are empty.

Also the lofar images are not present in the catalouge for night time, but still the new dataset has data for that particular times. So the noaa is also labelling that data timestamps. Ask pietro if i should exclude these data as they might not have relevant information. Ask why do we have the nighttime data for lofar??

 So look at the average start and end times of the sun images in the catalouge an remove the data images for the data for those nighttime images

The labelling with noaa is pretty decent for type 3 and 5. For type 2 it misses some images as the time range of the burst is a bit long. For type 4 its the worst as the time range in noaa is huge.

The problem with labelling that i have done in solar_events_cleaned analysis is that it wasnt able to label all the instances properly as sometimes the bursts recorded in noaa are longer than 15 minutes, which is the length of the lofar image. We can tackle this by using the start and end time in noaa burst and label all the image which are in this range. Which means that, for a parttcular date, we will start looking from the first image (we will not approximate the time in the timestamps) of lofar, and if an a burst from noaa comes in the 15 minute time ie between the time of two consecutive timestamps, we will label the image of that timestamp as a burst. But if the length of burst is longer than 15 minutes, then we will label all the timestamps (2 or even more), with the single noaa burst. 

This will help in preserving the original name of the timestamp as well as solve the problem of iregular labelling. 

Some type 4 are more than 7 h hours long in noaa but on inspecting lofar data, they are no seen the full period. Maybe to solve this we can use a time range only to label those images (for eg, for time reange of noaa burst more than 75 minutes, keep give labels only to the images to 5 images ie image timstamps of lofar in 75 minutes time), but still there's a possibility of not having the burst in the multiple labeled images. Like some will have a burst in those images but theres a chance so will not. Maybe we can solve this using MIL machine learning technique. 

For more images we can use GAN for the best representation of the burst images. Like we will plot all those images and see if we can manually select the best images for increasing dataset using GAN. 

### Lets label the noaa data from solar_events_cleaned file. 

In [1]:
import json
import pandas as pd
import re

# --- Fetch "date", "begin", "end", "particulars" from solar_events_cleaned.json ---
json_path = "USED Fetch data from NOAA (works)/solar_events_cleaned.json"

with open(json_path, "r") as f:
    events = json.load(f)

# If the JSON is a dict with a list inside, extract the list
if isinstance(events, dict):
    for v in events.values():
        if isinstance(v, list):
            events = v
            break

event_rows = []
for row in events:
    try:
        event_rows.append({
            "date": row["date"],
            "begin": row["begin"],
            "end": row["end"],
            "particulars": row["particulars"]
        })
    except KeyError:
        continue

events_df = pd.DataFrame(event_rows)
print("Events DataFrame:")
display(events_df.head())

# --- Fetch timestamps and keys from dates_and_urls_of_new_data.csv ---
csv_path = "dates_and_urls_of_new_data.csv"
timestamps_df = pd.read_csv(csv_path)

# Extract date, time, and keep the original key
def extract_date_time_key(s):
    # Example: 2044342_2024-08-14_11:20:00.000000
    match = re.search(r"(\d{4}-\d{2}-\d{2})[_T ](\d{2}:\d{2}:\d{2}(?:\.\d{1,6})?)", str(s))
    if match:
        return match.group(1), match.group(2), s
    return None, None, s

# Extract date, time, and keep the original key
def extract_date_time_key(s):
    # Example: 2044342_2024-08-14_11:20:00.000000
    match = re.search(r"(\d{4}-\d{2}-\d{2})[_T ](\d{2}:\d{2}:\d{2}(?:\.\d{1,6})?)", str(s))
    if match:
        return match.group(1), match.group(2), s
    return None, None, s

# Try to find the column with the timestamp key
timestamp_col = None
for col in timestamps_df.columns:
    if timestamps_df[col].astype(str).str.contains(r"\d{4}-\d{2}-\d{2}").any():
        timestamp_col = col
        break

if timestamp_col is None:
    raise ValueError("No column with timestamp found in CSV.")

timestamps_df[["date", "time", "key"]] = timestamps_df[timestamp_col].apply(lambda s: pd.Series(extract_date_time_key(s)))

# Sort by date and time
timestamps_df["date_dt"] = pd.to_datetime(timestamps_df["date"])
timestamps_df["time_dt"] = pd.to_timedelta(timestamps_df["time"])
timestamps_df = timestamps_df.sort_values(["date_dt", "time_dt"]).reset_index(drop=True)
timestamps_df = timestamps_df.drop(columns=["date_dt", "time_dt"])

print("Timestamps DataFrame (date, time, and key extracted, sorted):")
display(timestamps_df[["date", "time", "key"]].head())
print(f"Total timestamps: {len(timestamps_df)}")

Events DataFrame:


,date,begin,end,particulars
0,2022-05-02,0000,0240,III/1
1,2022-05-02,0249,0250,III/2
2,2022-05-02,0306,0306,III/1
3,2022-05-02,0342,0343,III/1
4,2022-05-02,0415,0415,III/2


Timestamps DataFrame (date, time, and key extracted, sorted):


,date,time,key
0,2022-05-02,06:30:00.000000,858918_2022-05-02_06:30:00.000000
1,2022-05-02,06:45:00.000000,858918_2022-05-02_06:45:00.000000
2,2022-05-02,07:00:00.000000,858918_2022-05-02_07:00:00.000000
3,2022-05-02,07:15:00.000000,858918_2022-05-02_07:15:00.000000
4,2022-05-02,07:30:00.000000,858918_2022-05-02_07:30:00.000000


Total timestamps: 49601


We will make some changes to the logic of the code. So we know that the timestamps_df is sequentially arranged by date and time. The interval between each instance of the timestamps_df in 15 minutes. So that means if we want to use the events_df which has labels (in particulars column) to label the instances of timestamps_df, using the dates and times values. 

So let's talk about an example to understand what we want the code to do. We want to label the instances using begin and end time of a solar burst (label) from the events_df to timestamps_df. SO if for date 2022-05-02, if a burst occurred at begin=1202 and end=1205, the timestamp which should be labeled in timestamps_df should be key=858918_2022-05-02_12:00:00.000000 since the burst occurs and ends under the 15 minutes from 12:00:000000. Since each timestamp has an corresponding image of which has the burst signal which is 15 minutes long.

Similarly, another example, for a burst in events_df, for date 2022-05-21, a burst happened at begin=1202 and end=1218, then the timestamps which should be labeled in timestamps_df are 861426_2022-05-21_12:00:00.000000 and 861426_2022-05-21_12:15:00.000000. We labeled them both for the same label of the same instance from events_df because the burst lasted for more than 15 minutes from 1202. If it was less then we only would have labeled 861426_2022-05-21_12:00:00.000000. Since the burst lasted longer than 15 minutes in this case, the corresponding images containing the burst will use up 2 timestamps because they will create 2 images. 

So the code should kind of look into each instances in the timestamp_df and label it one by if the burst occurs that timestamp or not because each timestamp has a 15 minute long image. And if the burst length is longer than 15 minutes then multiple timestamps should be labeled based on the begin and end time from the events_df. Now the code uses both begin and end instead of just begin to label the images

But also again make sure not to label too many timestamps as some of the burst last more than few hours. This will end up over labelling. So we still want to keep a threshold for the difference between begin and end time. If the difference is more than 75 minutes for a specific instance in events_df, then we would label only the timestamps which comes in those 75 minutes. You can maybe do this by making a variable in the for loop, that for the current instance in events_df, if the difference between end and begin is greater than 75, then only label the current timestamp which is about to be labeled plus the next 5 timestamps instances. Just make sure that there is double entry or duplicate entry in the timestamp_df otherwise it will label the same timestamp which has the same image of 15 minutes multiple times instead of labelling the next 15 minute timestamp. 

Save this list in a csv file with the begin and end times as well with dates from events_df which was used to label the timestamp

In [9]:
import pandas as pd

def label_timestamps(events_df, timestamps_df):
    """
    Match solar burst events to 15-minute timestamp intervals
    with special handling for long-duration events (>75 minutes)
    """
    # Create working copies to avoid modifying original DataFrames
    events_df = events_df.copy()
    timestamps_df = timestamps_df.copy()
    
    # Convert event times to proper datetime format
    events_df['begin_time'] = events_df['begin'].apply(lambda x: f"{x[:2]}:{x[2:]}")
    events_df['end_time'] = events_df['end'].apply(lambda x: f"{x[:2]}:{x[2:]}")
    events_df['begin_dt'] = pd.to_datetime(events_df['date'] + ' ' + events_df['begin_time'], errors='coerce')
    events_df['end_dt'] = pd.to_datetime(events_df['date'] + ' ' + events_df['end_time'], errors='coerce')
    
    # Create timestamp intervals (15-minute windows)
    timestamps_df['start_interval'] = pd.to_datetime(timestamps_df['date'] + ' ' + timestamps_df['time'])
    timestamps_df['end_interval'] = timestamps_df['start_interval'] + pd.Timedelta(minutes=15)
    
    # Prepare results storage
    results = []
    
    # Process each event
    for _, event in events_df.iterrows():
        # Skip events with invalid dates/times
        if pd.isnull(event['begin_dt']) or pd.isnull(event['end_dt']):
            continue
            
        event_date = event['date']
        event_begin = event['begin_dt']
        event_end = event['end_dt']
        duration = (event_end - event_begin).total_seconds() / 60
        
        # Skip negative durations
        if duration < 0:
            continue
            
        # Get timestamps for current date
        date_timestamps = timestamps_df[timestamps_df['date'] == event_date]
        if date_timestamps.empty:
            continue
            
        # Case 1: Long events (>75 minutes)
        if duration > 75:
            # Find starting timestamp interval
            start_mask = (
                (event_begin >= date_timestamps['start_interval']) & 
                (event_begin < date_timestamps['end_interval'])
            )
            start_intervals = date_timestamps[start_mask]
            
            if not start_intervals.empty:
                start_idx = start_intervals.index[0]
                date_indices = date_timestamps.index.tolist()
                
                # Get position of starting index
                try:
                    pos = date_indices.index(start_idx)
                    # Get next 5 intervals (total of 6 intervals)
                    end_pos = min(pos + 6, len(date_indices))
                    for i in range(pos, end_pos):
                        idx = date_indices[i]
                        results.append({
                            'timestamp_key': timestamps_df.at[idx, 'key'],
                            'timestamp_date': timestamps_df.at[idx, 'date'],
                            'timestamp_time': timestamps_df.at[idx, 'time'],
                            'event_date': event['date'],
                            'event_begin': event['begin'],
                            'event_end': event['end'],
                            'event_particulars': event['particulars']
                        })
                except ValueError:
                    pass
        
        # Case 2: Short/medium events (≤75 minutes)
        else:
            # Find all overlapping intervals
            overlap_mask = (
                (event_begin < date_timestamps['end_interval']) & 
                (event_end > date_timestamps['start_interval'])
            )
            overlapping = date_timestamps[overlap_mask]
            
            for idx, row in overlapping.iterrows():
                results.append({
                    'timestamp_key': row['key'],
                    'timestamp_date': row['date'],
                    'timestamp_time': row['time'],
                    'event_date': event['date'],
                    'event_begin': event['begin'],
                    'event_end': event['end'],
                    'event_particulars': event['particulars']
                })
    
    return pd.DataFrame(results)

# Load your data (replace with actual paths if reading from files)
events_df = events_df.copy()

timestamps_df = timestamps_df.copy()


# Process and save results
results_df = label_timestamps(events_df, timestamps_df)
results_df.to_csv('burst_event_matches.csv', index=False)

print(f"Saved {len(results_df)} matches to burst_event_matches.csv")
print("Sample results:")
print(results_df.head())

Saved 4325 matches to burst_event_matches.csv
Sample results:
                       timestamp_key timestamp_date   timestamp_time  \
0  858918_2022-05-02_06:45:00.000000     2022-05-02  06:45:00.000000   
1  858918_2022-05-02_09:15:00.000000     2022-05-02  09:15:00.000000   
2  858918_2022-05-02_09:30:00.000000     2022-05-02  09:30:00.000000   
3  858918_2022-05-02_10:15:00.000000     2022-05-02  10:15:00.000000   
4  859062_2022-05-03_07:00:00.000000     2022-05-03  07:00:00.000000   

   event_date event_begin event_end event_particulars  
0  2022-05-02        0649      0650             III/2  
1  2022-05-02        0929      0934             III/1  
2  2022-05-02        0929      0934             III/1  
3  2022-05-02        1020      1020             III/1  
4  2022-05-03        0704      0705             III/1  


#### Now add type 2.csv and previous data labels to complete the labelling and covert roman to ints

Convert roman to ints in burst_event_matches

In [1]:
import pandas as pd
import re

# Load the CSV
df = pd.read_csv('burst_event_matches.csv')

# Function to convert Roman numerals to integers
def roman_to_int(roman):
    roman_numeral_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5}
    match = re.match(r'([IVX]+)', roman)
    if match:
        return roman_numeral_map.get(match.group(1), None)
    return None

# Process event_particulars column: keep only the roman part and convert to int
def process_label(label):
    if pd.isnull(label):
        return None
    # Remove everything after "/" and convert roman to int
    roman_part = label.split('/')[0]
    return roman_to_int(roman_part)

# Apply conversion and rename column
df['labels'] = df['event_particulars'].apply(process_label)
df = df.drop(columns=['event_particulars'])

# Save to new CSV
df.to_csv('burst_event_matches_int.csv', index=False)

# Preview
print(df.head())

                       timestamp_key timestamp_date   timestamp_time  \
0  858918_2022-05-02_06:45:00.000000     2022-05-02  06:45:00.000000   
1  858918_2022-05-02_09:15:00.000000     2022-05-02  09:15:00.000000   
2  858918_2022-05-02_09:30:00.000000     2022-05-02  09:30:00.000000   
3  858918_2022-05-02_10:15:00.000000     2022-05-02  10:15:00.000000   
4  859062_2022-05-03_07:00:00.000000     2022-05-03  07:00:00.000000   

   event_date  event_begin  event_end  labels  
0  2022-05-02          649        650       3  
1  2022-05-02          929        934       3  
2  2022-05-02          929        934       3  
3  2022-05-02         1020       1020       3  
4  2022-05-03          704        705       3  


Append type 2 csv and previous labels. 

In [3]:
import csv
# Read dates_and_urls_of_new_data.csv
csv_file = 'dates_and_urls_of_new_data.csv'


type2_full_timestamps = set()
with open('type 2 list found in catalogue.csv', 'r') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  # Skip header row
    for row in csv_reader:
        if len(row) >= 3:
            date_time_str = row[1].strip()
            prefix = row[2].strip()
            # Parse date and time
            date_part, time_part = date_time_str.split(' ')
            day, month, year = date_part.split('/')
            year = f"20{year}"
            # Format as YYYY-MM-DD_HH:MM:00.000000
            formatted_key = f"{year}-{month}-{day}_{time_part}:00.000000"
            # Join prefix and formatted_key
            full_type2 = f"{prefix}_{formatted_key}"
            type2_full_timestamps.add(full_type2)

print(f"Type 2 full timestamps loaded: {(type2_full_timestamps)}")
print(f"Total type 2 timestamps: {len(type2_full_timestamps)}")


Type 2 full timestamps loaded: {'2029359_2024-05-04_06:26:00.000000', '2028693_2024-01-14_11:40:00.000000', '858338_2022-04-22_13:15:00.000000', '868448_2022-08-18_11:00:00.000000', '2022421_2023-07-15_10:05:00.000000', '881218_2023-02-09_08:56:00.000000', '2025205_2023-09-20_14:27:00.000000', '2040812_2024-05-29_14:35:00.000000', '867632_2022-08-03_17:00:00.000000', '2028255_2023-11-02_12:27:00.000000', '2040132_2024-05-14_17:11:00.000000', '2022439_2023-07-18_17:38:01:00.000000', '2028783_2024-01-29_10:25:00.000000', '2029359_2024-05-04_06:11:00.000000', '2028885_2024-02-15_08:59:00.000000', '881218_2023-02-09_07:26:00.000000', '2042370_2024-07-29_12:52:00.000000', '873300_2022-10-11_09:00:00.000000', '888144_2023-05-05_07:11:00.000000', '888168_2023-05-11_08:56:00.000000', '2028897_2024-02-17_13:24:00.000000', '862352_2022-05-28_15:45:00.000000', '2043569_2024-08-05_05:16:00.000000', '2022572_2023-07-24_18:00:00.000000', '2024373_2023-08-17_12:47:00.000000', '2040132_2024-05-14_17:4

In [8]:
import pandas as pd

# Load timestamps_df (make sure 'key' column exists)
timestamps_df = timestamps_df.copy()
# Compare type2_full_timestamps to timestamps_df['key']
matched_df = timestamps_df[timestamps_df['key'].isin(type2_full_timestamps)].copy()

# Manually add "2022439_2023-07-18_17:38:01.000000" as it had 01 in seconds
extra_row = pd.DataFrame([{
    'key': "2022439_2023-07-18_17:38:01.000000",
    'date': "18/07/23",
    'time': "17:38",
    'labels': 2
}])

matched_df = pd.concat([matched_df, extra_row], ignore_index=True)

# Add label column with value 2
matched_df['labels'] = 2

# Save or preview
matched_df.to_csv('type2_matched_labeled.csv', index=False)
print(matched_df[['key', 'date', 'time', 'labels']].head())
print(f"Total matched type 2 timestamps: {len(matched_df)}")

                                 key        date             time  labels
0  862222_2022-05-25_18:15:00.000000  2022-05-25  18:15:00.000000       2
1  862352_2022-05-28_15:45:00.000000  2022-05-28  15:45:00.000000       2
2  862352_2022-05-28_16:00:00.000000  2022-05-28  16:00:00.000000       2
3  867632_2022-08-03_17:00:00.000000  2022-08-03  17:00:00.000000       2
4  867632_2022-08-03_17:15:00.000000  2022-08-03  17:15:00.000000       2
Total matched type 2 timestamps: 124


In [9]:
# Print the keys in type2_full_timestamps that are NOT present in matched_df['key']
missing_keys = type2_full_timestamps - set(matched_df['key'])
print("Type 2 timestamp keys NOT found in timestamps_df:")
for key in missing_keys:
    print(key)

Type 2 timestamp keys NOT found in timestamps_df:
857860_2022-04-20_12:00:00.000000
2022439_2023-07-18_17:38:01:00.000000
858338_2022-04-22_13:15:00.000000
857860_2022-04-20_12:15:00.000000
858338_2022-04-22_13:30:00.000000


Lets add the previous data labels as well and save the burst_events_matches_int.csv to timestamps_labels.csv

In [17]:
import h5py
import pandas as pd

# Load timestamps_df (make sure 'key' column exists)
timestamps_df = timestamps_df.copy()

# Open your HDF5 file and extract keys and labels
h5_path = '/Volumes/External SSD 512gb/dset_with_labels.h5' 
with h5py.File(h5_path, 'r') as h5f:
    # Assuming the HDF5 file has datasets 'timestamps' and 'labels'
    h5_keys = [k.decode() if isinstance(k, bytes) else str(k) for k in h5f['timestamps'][:2854]]
    h5_labels = h5f['labels'][:2854]

# Build DataFrame from HDF5
h5_df = pd.DataFrame({'key': h5_keys, 'labels': h5_labels})

# Find overlaps with timestamps_df
overlap_df = timestamps_df[timestamps_df['key'].isin(h5_df['key'])].copy()

# Merge to get the labels from h5_df
overlap_df = overlap_df.merge(h5_df, on='key', how='left')

# Save or preview
overlap_df.to_csv('h5_labels_matched.csv', index=False)
print(overlap_df[['key', 'date', 'time', 'labels']].head())
print(f"Total matched HDF5 timestamps: {len(overlap_df)}")

                                 key        date             time  labels
0  858918_2022-05-02_06:30:00.000000  2022-05-02  06:30:00.000000       3
1  858918_2022-05-02_06:45:00.000000  2022-05-02  06:45:00.000000       3
2  858918_2022-05-02_07:00:00.000000  2022-05-02  07:00:00.000000       3
3  858918_2022-05-02_07:15:00.000000  2022-05-02  07:15:00.000000       3
4  858918_2022-05-02_07:30:00.000000  2022-05-02  07:30:00.000000       3
Total matched HDF5 timestamps: 2854


In [22]:
import pandas as pd

# Load CSVs
burst_df = pd.read_csv('burst_event_matches_int.csv')
type2_df = pd.read_csv('type2_matched_labeled.csv')
h5_df = pd.read_csv('h5_labels_matched.csv')

# --- Clean burst_events_matches_int.csv ---
burst_df = burst_df.drop(columns=['event_date', 'event_begin', 'event_end'], errors='ignore')
burst_df = burst_df.rename(columns={
    'timestamp_key': 'timestamps_key',
    'timestamp_date': 'date',
    'timestamp_time': 'time'
})
burst_df = burst_df[['timestamps_key', 'date', 'time', 'labels']]

# --- Clean type2_matched_labeled.csv ---
type2_df = type2_df.rename(columns={'Date': 'timestamps_key'})
type2_df = type2_df[['timestamps_key', 'date', 'time', 'labels']]

# --- Clean h5_labels_matched.csv ---
h5_df = h5_df.rename(columns={'Date': 'timestamps_key'})
h5_df = h5_df.drop(columns=['Index', 'URL', 'key'], errors='ignore')
h5_df = h5_df[['timestamps_key', 'date', 'time', 'labels']]

# --- Concatenate all ---
all_labels_df = pd.concat([burst_df, type2_df, h5_df], ignore_index=True)

# --- Save and print label counts ---
all_labels_df.to_csv('timestamps_labels.csv', index=False)

# Print the total number of labels
print(f"Total labels in all datasets: {len(all_labels_df)}")

# Print the number of unique timestamps
print(f"Total unique timestamps: {len(all_labels_df['timestamps_key'].unique())}")

# Print the count of each label
print("Label counts:")
print(all_labels_df['labels'].value_counts())

Total labels in all datasets: 7303
Total unique timestamps: 6640
Label counts:
labels
3    5142
6     615
1     559
2     422
4     363
5     202
Name: count, dtype: int64


There are still duplicate timestamps in the csv but we can easily remove and make unique timestamps using a dictionary format maybe